# [OPTIONAL] Data Augmentation for Computer Vision in PyTorch

<img src="http://pytorch.org/vision/master/_images/sphx_glr_plot_transforms_illustrations_012.png" alt="Data Augmentation for Computer Vision" width="1200" />


In this notebook, we will explore common **data augmentation techniques** for computer vision tasks.

Data augmentation serves several roles in training deep learning models:

- **Prevents Overfitting:** By increasing the amount and variability of training data, it helps models to not just memorize specific examples but to learn the underlying patterns, thus reducing overfitting.
- **Improves Generalization:** Augmentation helps the model generalize better by simulating a variety of scenarios that it might encounter in the real world.Transformations like scaling, rotating, and flipping help the model learn features that are *invariant* to these transformations, which is important in many computer vision tasks.
- **Enriches Small Datasets:** In situations with limited data, augmentation can artificially expand the size of the dataset, allowing for more robust learning.
- **Simulates Noise and Occlusions:** Introducing artificial noise or occlusions can make the model more resilient to such real-world issues. Occlusions in the context of computer vision refer to instances where a part of the subject (such as an object or a key feature) in an image is blocked from view, typically by another object.

We will be using `torchvision.transforms`, which offers a set of classes suitable for applying consistent transformations to images. In particular, we will use the latest version of these classes, contained in `torchvision.transforms.v2`.These classes can be chained together to create *transformation pipelines* with `transforms.Compose`.

Torchvision also provides `torchvision.transforms.functional`, which contains function-based transformations for more granular control, applied directly to images. However, the functional approach is typically unnecessary for standard applications because the predefined classes in `torchvision.transforms` offer a more straightforward and automated way to apply common transformations across entire datasets without the need for manual intervention.


This notebook takes inspiration from the official PyTorch documentation and examples, which can be found [here](http://pytorch.org/vision/master/auto_examples/transforms/plot_transforms_illustrations.html). Please refer to the PyTorch documeentation for further details.

## Load modules and connect to Google Drive

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import random
import pandas as pd
import os

import torch
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
from torchvision import tv_tensors
from torchvision.transforms import v2 as transforms
from torchvision.transforms.v2 import functional as F


In [ ]:
# Check if CUDA is available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load the UC Merced Land Use Dataset


In [ ]:
dataset_folder = 'data'

# Load CSV file and subsample (sample_size==-1 for full dataset)
df = pd.read_csv(os.path.join(dataset_folder, 'labels.csv'))

# Get image file paths and labels
image_paths = df['image'].values
labels = df['label'].values

# Load the mapping between integer labels and class names
label_mapping = {}
with open(os.path.join(dataset_folder, 'label_mapping.txt'), 'r') as file:
    for line in file:
        value, key = line.strip().split(': ')
        label_mapping[int(key)] = value

print(label_mapping)

## Select one image at random

For our data augmentation exercises, it is best to select an image that contains a variety of features and textures. Find a suitable image with a few random trials...

In [ ]:
# Randomly select an index
random_index = random.randint(0, len(image_paths) - 1)

# Use the index to retrieve the image path and label
selected_image_path = image_paths[random_index]
selected_label = label_mapping[labels[random_index]]

# Load your image (replace with your actual file path)
orig_img = Image.open(os.path.join(dataset_folder, f'Images/{selected_image_path}'))
plt.imshow(orig_img)
plt.gca().set_title(f'Original Image --- Class:{selected_label}');

# Geometric Augmentations in PyTorch

Geometric augmentations alter the spatial structure of the image. They are important for teaching our models about various spatial variances that could occur in real-world scenarios. These augmentations include operations like flipping, rotations, translations and scaling. By applying these transformations, we can train our models to be invariant to changes in position and orientation.

Let's begin by exploring the most commonly used geometric transformations provided by PyTorch, and understand how they can enhance our model's performance.


In [ ]:
def plot(imgs, row_title=None, **imshow_kwargs):
    """Plooting function taken from https://raw.githubusercontent.com/pytorch/vision/main/gallery/transforms/helpers.py"""
    if not isinstance(imgs[0], list):
        # Make a 2d grid even if there's just 1 row
        imgs = [imgs]

    num_rows = len(imgs)
    num_cols = len(imgs[0])
    _, axs = plt.subplots(nrows=num_rows, ncols=num_cols, squeeze=False)
    for row_idx, row in enumerate(imgs):
        for col_idx, img in enumerate(row):
            boxes = None
            masks = None
            if isinstance(img, tuple):
                img, target = img
                if isinstance(target, dict):
                    boxes = target.get("boxes")
                    masks = target.get("masks")
                elif isinstance(target, tv_tensors.BoundingBoxes):
                    boxes = target
                else:
                    raise ValueError(f"Unexpected target type: {type(target)}")
            img = F.to_image(img)
            if img.dtype.is_floating_point and img.min() < 0:
                # Poor man's re-normalization for the colors to be OK-ish. This
                # is useful for images coming out of Normalize()
                img -= img.min()
                img /= img.max()

            img = F.to_dtype(img, torch.uint8, scale=True)
            if boxes is not None:
                img = draw_bounding_boxes(img, boxes, colors="yellow", width=3)
            if masks is not None:
                img = draw_segmentation_masks(img, masks.to(torch.bool), colors=["green"] * masks.shape[0], alpha=.65)

            ax = axs[row_idx, col_idx]
            ax.imshow(img.permute(1, 2, 0).numpy(), **imshow_kwargs)
            ax.set(xticklabels=[], yticklabels=[], xticks=[], yticks=[])

    if row_title is not None:
        for row_idx in range(num_rows):
            axs[row_idx, 0].set(ylabel=row_title[row_idx])

    plt.tight_layout()

### Flipping

The `transforms.functional.hflip` and `transforms.functional.vflip` functions in `torchvision.transforms.functional` offer a straightforward approach to image augmentation through deterministic horizontal and vertical flipping. These functions mirror the image across the horizontal or vertical axis.

In [ ]:
plot([orig_img] + [F.hflip(orig_img),F.vflip(orig_img)])

In practice, we typically employ random flipping for data augmentation, utilizing `transforms.RandomHorizontalFlip` and `transforms.RandomVerticalFlip`. These transformations introduce variability into the dataset by flipping images horizontally or vertically with a certain probability `p`. The parameter `p` dictates the likelihood of each image being flipped; setting `p` to 1 would replicate the behavior of deterministic flips, as in `transforms.functional.hflip` and `transforms.functional.vflip`, flipping every image in the dataset. Random flipping is crucial for training models to be robust to orientation changes, simulating a more diverse set of real-world scenarios.

In [ ]:
plot([orig_img] + [transforms.RandomHorizontalFlip(p=1)(orig_img),transforms.RandomVerticalFlip(p=1)(orig_img)])

### Padding

The `transforms.Pad` transformation in PyTorch's `torchvision.transforms` module is used to add padding around the image. The `padding` argument specifies the number of pixels to pad on each border. Padding can be a useful preprocessing step, particularly when you want to maintain a certain image size after cropping or other transformations. However, it's not always beneficial; for instance, indiscriminate padding might introduce too much non-informative space into the image, which could potentially affect the model's learning efficacy. See below how the black background may affect the information in the original image.

In [ ]:
padded_imgs = [transforms.Pad(padding=padding)(orig_img) for padding in (3, 10, 30, 50)]
plot([orig_img] + padded_imgs)

Also, by using `transforms.Pad`, the size of the images increases by twice the number of padding pixels, expanding the original dimensions by adding the specified padding around each edge of the image.


In [ ]:
padded_imgs

### Resize

`transforms.Resize` rescales an image to the given `size`, which can be specified as a desired height and width or a single number to maintain aspect ratio. This transformation is essential for standardizing input image sizes, ensuring consistency across a dataset before feeding it into a neural network.


In [ ]:
resized_imgs = [transforms.Resize(size=size)(orig_img) for size in (30, 50, 100, orig_img.size)]
plot([orig_img] + resized_imgs)

Notice how the image size changed, according to what we specified

In [ ]:
resized_imgs

### CenterCrop and FiveCrop

`transforms.CenterCrop` is a transformation that crops the given image at the center to the specified `size`. This is particularly useful for focusing on the central region of an image, which often contains the subject of interest in many computer vision tasks.

On the other hand, `transforms.FiveCrop` generates five crops from the given image — the four corners and the central crop. This helps in data augmentation by providing multiple perspectives of the same image, potentially increasing the robustness of the model to variations in object placement within the image.


In [ ]:
center_crops = [transforms.CenterCrop(size=size)(orig_img) for size in (30, 50, 100, orig_img.size)]
plot([orig_img] + center_crops)

In [ ]:
center_crops

In [ ]:
(top_left, top_right, bottom_left, bottom_right, center) = transforms.FiveCrop(size=(100, 100))(orig_img)
plot([orig_img] + [top_left, top_right, bottom_left, bottom_right, center])

### RandomCrop and RandomResizedCrop

`transforms.RandomCrop` takes a random cut of the specified `size` from the input image, which can remove parts of the image's content and encourages the model to focus on different image regions during training. This process can also help the model in recognizing objects even when they are partially visible. `transforms.RandomResizedCrop` combines cropping and resizing, extracting a random portion of the image and then resizing it back to a specified `size`.

Excessive cropping could lead to a situation where the remaining portion of the image no longer represents the original label, potentially confusing the model during training.

In [ ]:
cropper = transforms.RandomCrop(size=(128, 128))
crops = [cropper(orig_img) for _ in range(4)]
plot([orig_img] + crops)

In [ ]:
crops

In [ ]:
resize_cropper = transforms.RandomResizedCrop(size=(32, 32))
resized_crops = [resize_cropper(orig_img) for _ in range(4)]
plot([orig_img] + resized_crops)

In [ ]:
resized_crops

### RandomPerspective

`transforms.RandomPerspective` applies a random perspective transformation to the image with a specified `distortion_scale` controlling the degree of distortion and `p` indicating the probability of the transformation being applied. This is important as it mimics real-world variations due to changes in viewpoint, enhancing the model's ability to understand perspective changes.

This transformation can create areas within the transformed image that were not part of the original, resulting in extra background that might not be informative and could potentially mislead the model if not handled properly.

In [ ]:
perspective_transformer = transforms.RandomPerspective(distortion_scale=0.6, p=1.0)
perspective_imgs = [perspective_transformer(orig_img) for _ in range(4)]
plot([orig_img] + perspective_imgs)

### RandomRotation

`transforms.RandomRotation` rotates the image by a random angle within the specified `degrees` range. This transformation is significant for training models to recognize objects regardless of orientation, which is a common variation in real-world scenarios.

Even in this case, a potential issue with this approach is that it can introduce artificial background areas into the rotated image, especially if the rotation angle is large, which may not be representative of the actual scene and could affect the model's learning.


In [ ]:
rotater = transforms.RandomRotation(degrees=(0, 180))
rotated_imgs = [rotater(orig_img) for _ in range(4)]
plot([orig_img] + rotated_imgs)

### RandomAffine

An *affine transformation* in geometry is a linear mapping method that preserves points, straight lines, and planes. Sets of parallel lines remain parallel after an affine transformation. In computer vision, `transforms.RandomAffine` performs such transformations on images, including random rotations (within the specified `degrees`), translations (shifts indicated by `translate`), and scaling (controlled by `scale`). These affine transformations are crucial for teaching models to recognize and understand objects in images that have undergone changes in position and/or orientation and/or size.

However, similar to other transformations, `transforms.RandomAffine` can introduce empty areas into the image, typically filled with a default color like black, which may not correspond to any meaningful content and could potentially disrupt the model's training.

In [ ]:
affine_transfomer = transforms.RandomAffine(degrees=(30, 70), translate=(0.1, 0.3), scale=(0.5, 0.75))
affine_imgs = [affine_transfomer(orig_img) for _ in range(4)]
plot([orig_img] + affine_imgs)

### Taking care of the added background...

There are different ways to take care of the problems introduced by the added *background* of some geometrical transformations.

One such approach is performing a *largest contained crop*, that is cropping the image to the largest area that does not include the artificially introduced background. However, this is not a built-in feature in PyTorch's `torchvision.transforms`, and it typically requires a custom implementation. This task is especially complex following rotations or skewing, where the image's original corners no longer align with the axes.

A more straightforward method is to apply `transforms.CenterCrop` post-transformation, which crops the image's center. This technique presumes that the image's central region is most likely to be free of the background, a fair assumption for several affine transformations, particularly if padding was added initially to compensate for potential rotation or scaling.

The following code snippet will illustrate this concept with a deterministic rotation set at 30 degrees (i.e., `degrees=(30,30)`). Observe how applying `transforms.CenterCrop` with varying sizes influences the presence of the background in the final output.

In [ ]:
rotater = transforms.RandomRotation(degrees=(30,30))
rotated_imgs = [transforms.CenterCrop(size=size)(rotater(orig_img)) for size in [256,200,180,128]]
plot([orig_img] + rotated_imgs)

In [ ]:
rotated_imgs

## Photometric Transforms

# Photometric Transforms in PyTorch

While geometric transformations modify the spatial arrangement of pixels in images, photometric transforms alter the color properties and lighting conditions without changing the image structure. These types of transformations are important for simulating various lighting and color scenarios that a model might encounter, enhancing the robustness against changes in illumination and color variations.

- **`transforms.Grayscale`**: Converts the image to grayscale, reducing it to shades of gray, effectively removing color information. This is particularly useful when color is not a defining characteristic for the task.

- **`transforms.ColorJitter`**: Adjusts the brightness and hue of the image to simulate different lighting conditions and color variations. The `brightness` parameter controls how much the image's brightness is altered, and the `hue` parameter adjusts the color hue. This transform can make a model more resilient to changes in illumination and color distribution.

- **`transforms.GaussianBlur`**: Applies a Gaussian blur to the image with a defined `kernel_size` and `sigma` (standard deviation). Blurring can simulate the effect of camera focus variations and atmospheric conditions.

There are numerous other photometric transformations available in `torchvision`, such as contrast adjustment, saturation changes, and many more. For a comprehensive list and detailed explanations, students are encouraged to refer to the [torchvision documentation](https://pytorch.org/vision/stable/transforms.html).


### Grayscale

In [ ]:
gray_img = transforms.Grayscale()(orig_img)
plot([orig_img, gray_img], cmap='gray')

### ColorJitter

In [ ]:
jitter = transforms.ColorJitter(brightness=.5, hue=.3)
jittered_imgs = [jitter(orig_img) for _ in range(4)]
plot([orig_img] + jittered_imgs)

### GaussianBlur

In [ ]:
blurrer = transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.))
blurred_imgs = [blurrer(orig_img) for _ in range(4)]
plot([orig_img] + blurred_imgs)

# Building Transformation Pipelines




`transforms.Compose` is a PyTorch utility that allows for the combination of multiple image transformations into a single pipeline. This pipeline can then be applied to images in a consistent and efficient manner.

Below, we will demonstrate two examples of transformation pipelines:

- **Simple Pipeline**: Our first example is straightforward, featuring a double flip - first horizontally, then vertically. This helps the model learn to recognize images regardless of their orientation.

- **Complex Pipeline**: The second example is more intricate, encompassing a series of transformations including flipping, rotations, blurring and random crops. This exposes the model to a broader range of variations.

When constructing these pipelines, it's crucial to be judicious in selecting which transformations to apply. Each added transformation introduces a new layer of complexity to the learning process. **It's important to choose transformations that add value and align with the real-world conditions the model will encounter**. Overcomplicating the pipeline with too many or irrelevant transformations can be counterproductive, potentially confusing the model and hindering performance.


In [ ]:
# Define the transformation pipeline with flipping
transformation_pipeline = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1),
    transforms.RandomVerticalFlip(p=1)  # Crop without resizing
])

transformed_imgs = [transformation_pipeline(orig_img) for _ in range(1)]
plot([orig_img] + transformed_imgs)

In [ ]:
# Define the transformation pipeline with flipping, rotation, blur and random resized crop
transformation_pipeline = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.2),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=(0, 180)),
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)),
    transforms.RandomResizedCrop(size=(180, 180))
])

transformed_imgs = [transformation_pipeline(orig_img) for _ in range(5)]
plot([orig_img] + transformed_imgs)